# CPC–xLSTM con validación *leave-one-subject-out* (LOSO)

**Proyecto Integrador — Equipo 1**

Este notebook reevalúa las configuraciones finalistas de `06_CPC_fs_configurable.ipynb` con **validación cruzada dejando un sujeto fuera** (*leave-one-subject-out*, LOSO).

## ¿Por qué LOSO?

En el 06, todas las decisiones (frecuencia, modelo, horizonte, representación) se tomaron con **2 sujetos de validación** (11 y 12). La diferencia entre esos dos sujetos (F1 ≈ 0.60 contra ≈ 0.48 en el ajuste fino) es mayor que la diferencia entre configuraciones (0.01–0.03). Con dos personas no se puede saber si una ventaja se mantendría con otras.

**LOSO** repite el experimento una vez por sujeto:

- en cada **pliegue**, un sujeto queda fuera y se usa sólo para evaluar;
- el modelo se entrena con los demás;
- al final se tiene un F1 **por persona**.

La media de esos valores estima el desempeño con personas nuevas, y su dispersión indica **cuánto varía de una persona a otra**.

## Diseño

| Decisión | Implementación | Por qué |
| :--- | :--- | :--- |
| Sujetos | Desarrollo = 1–12 (12 pliegues). **13 y 14 nunca se usan** | La prueba sigue cerrada hasta congelar todas las decisiones. |
| Preentrenamiento CPC | Con los sujetos de entrenamiento del pliegue; el sujeto evaluado **no** se ve, ni siquiera sin etiquetas | Usar su señal sin etiquetas también sería fuga de información. |
| Escalador | `clip + log1p + RobustScaler` **reajustado en cada pliegue** con los sujetos de entrenamiento | El escalador exportado se ajustó con los sujetos 1–10, que incluyen sujetos evaluados aquí. |
| Número de épocas | **Fijo**: la mediana de las mejores épocas del 06 a 12 Hz | Sin un sujeto de validación aparte no hay parada temprana honesta. Como esas épocas se eligieron mirando a los sujetos 11 y 12, sus pliegues pueden salir algo optimistas (ver sección 6). |
| Configuraciones | Las 2 finalistas del 06 a 12 Hz (`mismo_sujeto`) | El barrido completo × 12 pliegues sería prohibitivo; LOSO se usa para **confirmar** la elección, no para buscarla. |
| Vistas por pliegue | Sonda lineal, ajuste fino **preentrenado** y **desde cero**, red **sin entrenar** y línea base **tabular** | Permite medir, persona por persona, qué aporta CPC y qué aporta el ajuste fino. |

**Requisito:** la exportación de `avance2.equipo1.ipynb` a la frecuencia elegida (`notebooks/_cache/avance2/fs12/`). Se genera ejecutando el avance 2 con `TARGET_FS = 12` y descomentando `export_artifacts()` en su sección 6.

**Duración (estimada en MPS):** ~15–25 min por pliegue y configuración con las tres variantes; ~8 h para 12 pliegues × 2 configuraciones. Cada resultado se guarda al terminar: si se interrumpe, al volver a ejecutar sólo se calcula lo que falta. `HELD_OUT` y `CONFIGS` permiten correr por partes.

# 1. Configuración

Las configuraciones finalistas y sus épocas salen de las corridas del 06 a 12 Hz (3 semillas cada una):

| Configuración | Modelo | Horizonte | Representación | Épocas CPC (mejores en el 06) | Épocas ajuste fino preentrenado / desde cero |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `mLSTM_A` | mLSTM | 0.5 s (K = 6 a 12 Hz) | `c_T+media` | 12 (20, 4, 12) | 2 (2, 1, 4) / 6 (6, 8, 4) |
| `sLSTM_B` | sLSTM | 1.0 s (K = 12 a 12 Hz) | `c_T` | 8 (8, 16, 4) | 5 (4, 8, 5) / 6 (6, 8, 5) |

Con `QUICK = True` se hace una corrida de humo (un pliegue, pocas iteraciones) en una carpeta aparte, para comprobar que todo funciona antes de lanzar el experimento completo.

In [ ]:
import copy, json, math, time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

# ----------------------------- Experimento -----------------------------
TARGET_FS = 12                          # debe existir notebooks/_cache/avance2/fs{TARGET_FS}/
CHANNEL_MODE = "8"                      # "8" o "pares4" (promedio de ch1–2, ch3–4, ch5–6, ch7–8)
DEV_SUBJECTS = tuple(range(1, 13))      # sujetos de desarrollo; 13 y 14 (prueba) nunca se usan
HELD_OUT = DEV_SUBJECTS                 # pliegues a ejecutar (se puede acotar para correr por partes)
CONFIGS = {
    "mLSTM_A": {"kind": "mLSTM", "horizon_s": 0.5, "batch_mode": "mismo_sujeto", "rep": "c_T+media",
                "cpc_epochs": 12, "ft_epochs": {"preentrenado": 2, "desde_cero": 6}},
    "sLSTM_B": {"kind": "sLSTM", "horizon_s": 1.0, "batch_mode": "mismo_sujeto", "rep": "c_T",
                "cpc_epochs": 8, "ft_epochs": {"preentrenado": 5, "desde_cero": 6}},
}
VARIANTS = ("sin_entrenar", "sonda", "preentrenado", "desde_cero")
SEED = 0
QUICK = False                           # True = corrida de humo (1 pliegue, pocas iteraciones, carpeta aparte)

# ----------------------------- Hiperparámetros (los mismos del 06) -----------------------------
MAX_PRED_HEADS = 12
CPC_SAMPLING = "recortes_aleatorios"
BATCH_SIZE, LR, GRAD_CLIP = 128, 5e-4, 1.0
PROBE_TRAIN_MAX = 15_000
FINETUNE_BATCH, FINETUNE_LR_HEAD, FINETUNE_LR_BACKBONE = 256, 1e-3, 1e-4
SCALER_Q, SCALER_FIT_MAX = (0.001, 0.999), 250_000
RERUN = False                           # True = repetir aunque exista el resultado
DEVICE_OVERRIDE = None

STEPS_CAP = FT_TRAIN_CAP = EVAL_CAP = None
if QUICK:
    HELD_OUT, STEPS_CAP, PROBE_TRAIN_MAX, FT_TRAIN_CAP, EVAL_CAP = (11,), 3, 2_000, 1_500, 1_500
    CONFIGS = {name: {**cfg, "cpc_epochs": 1, "ft_epochs": {"preentrenado": 1, "desde_cero": 1}} for name, cfg in CONFIGS.items()}
# -----------------------------------------------------------------------------------------------

if DEVICE_OVERRIDE:
    DEVICE = torch.device(DEVICE_OVERRIDE)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
EXPORT_DIR = REPO_DIR / "notebooks" / "_cache" / "avance2" / f"fs{TARGET_FS}"
TAG = f"fs{TARGET_FS}" + ("" if CHANNEL_MODE == "8" else f"_{CHANNEL_MODE}") + ("_quick" if QUICK else "")
RUN_DIR = REPO_DIR / "checkpoints" / f"cpc06_loso_{TAG}"
SUMMARY_CSV = (RUN_DIR if QUICK else REPO_DIR / "notebooks" / "_cache") / f"cpc06_loso_{TAG}_resultados.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert (EXPORT_DIR / "metadata.json").exists(), (
    f"Falta {EXPORT_DIR}: ejecutar avance2.equipo1.ipynb con TARGET_FS = {TARGET_FS} y export_artifacts()")
assert set(HELD_OUT) <= set(DEV_SUBJECTS), "Sólo se dejan fuera sujetos de desarrollo"


def horizons_for(seconds, fs=TARGET_FS, max_heads=MAX_PRED_HEADS):
    """Horizontes k (en pasos) hasta `seconds`. Consecutivos si caben en `max_heads`; si no, espaciados."""
    n = max(1, int(round(seconds * fs)))
    if n <= max_heads:
        return tuple(range(1, n + 1))
    return tuple(int(k) for k in np.unique(np.round(np.linspace(n / max_heads, n, max_heads)).astype(int)))


for cfg in CONFIGS.values():
    cfg["horizons"] = horizons_for(cfg["horizon_s"])
print("PyTorch:", torch.__version__, "| dispositivo:", DEVICE, "| QUICK:", QUICK)
print("Pliegues:", HELD_OUT, "| resultados en", RUN_DIR.relative_to(REPO_DIR))
display(pd.DataFrame({name: {"modelo": c["kind"], "horizontes k": c["horizons"], "batch": c["batch_mode"],
                             "representación": c["rep"], "épocas CPC": c["cpc_epochs"],
                             "épocas ajuste fino": c["ft_epochs"]} for name, c in CONFIGS.items()}).T)

# 2. Datos exportados por el avance 2

Se lee la serie continua **sin escalar** (`%MVC`), las etiquetas por muestra, los índices de ventanas y las 202 características. El escalado se hace después, **dentro de cada pliegue** (sección 4), con los sujetos de entrenamiento de ese pliegue.

In [ ]:
meta = json.loads((EXPORT_DIR / "metadata.json").read_text())
assert meta["target_fs"] == TARGET_FS and meta["label_rule"] == "final_instant"
with np.load(EXPORT_DIR / "series.npz", allow_pickle=False) as stored:
    signals_raw, offsets = stored["signals"].astype(np.float32), stored["offsets"]

trials = pd.read_csv(EXPORT_DIR / "trials.csv")
cpc_index = pd.read_csv(EXPORT_DIR / "cpc_index.csv.gz")
sup_index = pd.read_csv(EXPORT_DIR / "supervised_index.csv.gz")
features = pd.read_csv(EXPORT_DIR / "features.csv.gz")
assert (trials["offset"].to_numpy() == offsets[:-1]).all()

WIN, TAIL, LABELS = meta["window_len"], meta["tail_steps"], meta["label_order"]
N_CH = 8 if CHANNEL_MODE == "8" else 4
FEATURE_COLUMNS = meta["feature_columns"]
feat_lookup = features.set_index(["trial_idx", "end_step"])[FEATURE_COLUMNS]
WIN_RANGE = torch.arange(WIN, device=DEVICE)
SIGNALS_T = None                                       # serie escalada del pliegue actual (canales, muestras)

counts = (sup_index[sup_index["subject"].isin(DEV_SUBJECTS)]
          .pivot_table(index="subject", columns="target", values="end_step", aggfunc="size", fill_value=0)
          .rename(columns=dict(enumerate(LABELS))))
print(f"{TARGET_FS} Hz | ventana {WIN} pasos ({WIN / TARGET_FS:g} s) | cola {TAIL} pasos | canales {N_CH} ({CHANNEL_MODE})")
print("Ventanas supervisadas por sujeto de desarrollo (cada fila es un pliegue):")
display(counts.assign(total=counts.sum(axis=1)))

# 3. Modelo (idéntico al notebook 06)

Las celdas siguientes son **copia exacta** de las del 06: `CPCConfig`, el encoder causal, la GRU, el predictor, InfoNCE, `CPCModel`, sLSTM, mLSTM, el `g_ar` xLSTM, `build_model` y `summarize`. Así cualquier diferencia de resultados se debe al protocolo de evaluación, no al modelo.

In [ ]:
@dataclass
class CPCConfig:
    """
    Hiperparámetros de CPC (los mismos del notebook 01), más `pred_horizons`:
    la lista de horizontes k que se predicen. Vacía = k = 1..n_pred_steps, el comportamiento original.
    """
    # --- entrada ---
    n_channels: int = 6

    # --- g_enc: encoder convolucional ---
    enc_channels: tuple = (32, 64, 128)
    enc_kernel_size: int = 3
    enc_dropout: float = 0.2

    # --- g_ar: modelo autorregresivo ---
    ar_hidden_size: int = 256
    ar_num_layers: int = 2
    ar_dropout: float = 0.2

    # --- predicción / pérdida contrastiva ---
    n_pred_steps: int = 12                  # K: número de horizontes
    n_anchors: int = 4                      # nº de instantes de anclaje por batch de entrenamiento
    pred_horizons: tuple = ()               # NUEVO: horizontes k (en pasos); vacío = 1..n_pred_steps

    @property
    def horizons(self) -> tuple:
        return tuple(self.pred_horizons) or tuple(range(1, self.n_pred_steps + 1))

    @property
    def z_dim(self) -> int:
        return self.enc_channels[-1]

    @property
    def c_dim(self) -> int:
        return self.ar_hidden_size

In [ ]:
class CausalEncoder(nn.Module):
    """g_enc causal: bloques Conv1d -> ReLU -> Dropout sin stride (Haresamudram et al., 2020),
    con relleno sólo a la izquierda: z_t depende de x_{t-3..t}, nunca del futuro.

        x: (B, n_channels, T)  -->  z: (B, z_dim, T)
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        k = cfg.enc_kernel_size
        self.left_pad = k - 1
        in_channels = (cfg.n_channels,) + cfg.enc_channels[:-1]
        self.convs = nn.ModuleList([nn.Conv1d(c_in, c_out, kernel_size=k) for c_in, c_out in zip(in_channels, cfg.enc_channels)])
        self.dropout = nn.Dropout(cfg.enc_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = x
        for conv in self.convs:
            z = self.dropout(F.relu(conv(F.pad(z, (self.left_pad, 0)))))
        return z

In [ ]:
class AutoRegressive(nn.Module):
    """
    g_ar: modelo autorregresivo genérico (GRU multicapa).

        z: (B, L, z_dim)  -->  c: (B, L, c_dim)

    Los parámetros de GRU se especifican en la clase cfg. De acuerdo al paper: 
    - input_size: 128
    - hidden_Size: 256
    - num_layer: 2

    c[:, t] resume z[:, :t+1] de forma causal (una GRU nunca mira el futuro),
    que es la propiedad necesaria para usar c_t como predictor de z_{t+k}.

    Arquitectura (Haresamudram, Essa & Plötz, 2020): GRU de 2 capas con 256
    unidades ocultas y dropout entre capas.
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.gru = nn.GRU(
            input_size=cfg.z_dim,
            hidden_size=cfg.ar_hidden_size,
            num_layers=cfg.ar_num_layers,
            dropout=cfg.ar_dropout if cfg.ar_num_layers > 1 else 0.0,  # PyTorch exige 0 si num_layers==1
            batch_first=True,
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        c, _ = self.gru(z)        # c: (B, L, c_dim) - Hace un loop sobre todo el eje temporal toma el c_{t-1} e incorpora el nuevo paso z_t
        return c

In [ ]:
class Predictor(nn.Module):
    """
    Cabezas de predicción log-bilineal: una W_k (sin sesgo) por cada horizonte de cfg.horizons.

        c_t: (B, c_dim) -> (n_horizontes, B, z_dim),  con pred_k = W_k c_t ≈ z_{t+k}
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.horizons = cfg.horizons
        self.heads = nn.ModuleList([nn.Linear(cfg.c_dim, cfg.z_dim, bias=False) for _ in self.horizons])

    def forward(self, c_t: torch.Tensor) -> torch.Tensor:
        return torch.stack([head(c_t) for head in self.heads], dim=0)


class InfoNCE(nn.Module):
    """
    Pérdida contrastiva de CPC (Ec. 4) con negativos de minibatch, para cada horizonte de la lista.
    Para el horizonte k, el positivo de la secuencia i es z_{t+k}^(i) y los negativos son los
    z_{t+k}^(j) (j != i) de las otras secuencias del batch.
    """

    def __init__(self, horizons: tuple):
        super().__init__()
        self.horizons = tuple(horizons)

    def forward(self, preds: torch.Tensor, z: torch.Tensor, t: int):
        """
        preds: (n_horizontes, B, z_dim) · z: (B, L, z_dim) · t: instante de anclaje
        Devuelve (loss_mean, loss_per_k, acc_per_k), los dos últimos de forma (n_horizontes,).
        """
        H, B, _ = preds.shape
        target = torch.arange(B, device=preds.device)
        loss_per_k = torch.zeros(H, device=preds.device)
        acc_per_k = torch.zeros(H, device=preds.device)
        for j, k in enumerate(self.horizons):
            logits = preds[j] @ z[:, t + k, :].t()          # (B, B): S_ij = pred_i · z_{t+k}^(j)
            loss_per_k[j] = F.cross_entropy(logits, target)
            acc_per_k[j] = (logits.argmax(dim=1) == target).float().mean()
        return loss_per_k.mean(), loss_per_k, acc_per_k.detach()


class CPCModel(nn.Module):
    """Modelo CPC completo (Encoder + g_ar + Predictor + InfoNCE), con horizontes de cfg.horizons."""

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = CausalEncoder(cfg)          # encoder causal (sección 3.1)
        self.ar = AutoRegressive(cfg)
        self.predictor = Predictor(cfg)
        self.info_nce_loss = InfoNCE(cfg.horizons)

    def encode(self, x: torch.Tensor):
        """x: (B, n_channels, T) -> z: (B, T, z_dim), c: (B, T, c_dim)"""
        z = self.encoder(x).transpose(1, 2)
        return z, self.ar(z)

    def anchor_range(self, L: int):
        lo, hi = max(1, L // 8), L - max(self.cfg.horizons)   # deja sitio para leer z_{t+k_máx}
        if hi <= lo:
            raise ValueError(f"Ventana T={L} demasiado corta para el horizonte {max(self.cfg.horizons)}")
        return lo, hi

    def info_nce(self, x: torch.Tensor, anchors=None):
        z, c = self.encode(x)
        if anchors is None:
            anchors = torch.randint(*self.anchor_range(z.shape[1]), size=(self.cfg.n_anchors,)).tolist()
        losses, accs = [], []
        for t in anchors:
            _, loss_per_k, acc_per_k = self.info_nce_loss(self.predictor(c[:, t, :]), z, t)
            losses.append(loss_per_k)
            accs.append(acc_per_k)
        loss_per_k = torch.stack(losses).mean(dim=0)
        acc_per_k = torch.stack(accs).mean(dim=0)
        return loss_per_k.mean(), loss_per_k, acc_per_k

In [ ]:
class BlockDiagonalLinear(nn.Module):
    """
    Capa lineal con matriz de pesos block-diagonal: cada "head" (bloque) solo mezcla sus propias
    dimensiones. Es la forma en que xLSTM limita la mezcla entre celdas (sLSTM) o abarata las
    proyecciones q, k, v (mLSTM).

        x: (..., dim) -> (..., dim)     con n_blocks bloques de tamaño dim // n_blocks
    """

    def __init__(self, dim: int, n_blocks: int, bias: bool = True):
        super().__init__()
        assert dim % n_blocks == 0, "dim debe ser múltiplo de n_blocks"
        self.n_blocks, self.block = n_blocks, dim // n_blocks
        self.weight = nn.Parameter(torch.randn(n_blocks, self.block, self.block) / math.sqrt(self.block))
        self.bias = nn.Parameter(torch.zeros(dim)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shape = x.shape
        x = x.reshape(*shape[:-1], self.n_blocks, self.block)
        y = torch.einsum("...nd,ned->...ne", x, self.weight).reshape(shape)
        return y if self.bias is None else y + self.bias

In [ ]:
class sLSTMCell(nn.Module):
    """
    Celda sLSTM (Beck et al. 2024, Ec. 34-40, con estabilización Ec. 15-17).

        x: (B, T, dim) -> h: (B, T, dim)

    Diferencias con la LSTM clásica:
      - compuerta de entrada EXPONENCIAL  i_t = exp(ĩ_t)  (permite "revisar" decisiones de memoria)
      - estado normalizador  n_t = f_t n_{t-1} + i_t,  y  h_t = o_t * c_t / n_t
      - estado estabilizador m_t para que exp() no desborde (no cambia el resultado)
      - memory mixing por heads: la recurrencia R h_{t-1} es block-diagonal (n_heads bloques)

    Es recurrente (no paralelizable): recorre el tiempo paso a paso, igual que la GRU.
    """

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.dim = dim
        self.W = nn.Linear(dim, 4 * dim)                                     # entrada -> z, i, f, o
        self.R = nn.ModuleList([BlockDiagonalLinear(dim, n_heads, bias=False) for _ in range(4)])
        with torch.no_grad():                                                # sesgo de la forget gate en [3, 6]
            self.W.bias[2 * dim:3 * dim] = torch.linspace(3.0, 6.0, dim)    # (Tabla 2 del paper): al inicio "recuerda"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        Wx = self.W(x)                                                       # parte de la entrada, para todo t a la vez
        h = c = n = m = x.new_zeros(B, D)
        outputs = []
        for t in range(T):
            Rh = torch.cat([r(h) for r in self.R], dim=-1)                   # parte recurrente (memory mixing)
            z_pre, i_pre, f_pre, o_pre = (Wx[:, t] + Rh).chunk(4, dim=-1)
            log_f = F.logsigmoid(f_pre)                                      # forget gate sigmoide, en log
            m_new = torch.maximum(log_f + m, i_pre)                          # Ec. 15: estabilizador
            i_gate = torch.exp(i_pre - m_new)                                # Ec. 16: i' (exponencial estabilizada)
            f_gate = torch.exp(log_f + m - m_new)                            # Ec. 17: f'
            c = f_gate * c + i_gate * torch.tanh(z_pre)                      # Ec. 34: estado de celda
            n = f_gate * n + i_gate                                          # Ec. 35: normalizador
            h = torch.sigmoid(o_pre) * c / n                                 # Ec. 36: estado oculto
            m = m_new
            outputs.append(h)
        return torch.stack(outputs, dim=1)

In [ ]:
class sLSTMBlock(nn.Module):
    """
    Bloque residual sLSTM con "post up-projection" (Fig. 10 del paper, sin la convolución opcional):

        x -> LayerNorm -> sLSTM (n_heads) -> GroupNorm -> (+x)
          -> LayerNorm -> MLP con compuerta (factor 4/3, GeLU) -> (+x)
    """

    def __init__(self, dim: int, n_heads: int, dropout: float, proj_factor: float = 4 / 3):
        super().__init__()
        inner = int(round(proj_factor * dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.cell = sLSTMCell(dim, n_heads)
        self.gn = nn.GroupNorm(n_heads, dim)                                 # LayerNorm por head
        self.up_a, self.up_b = nn.Linear(dim, inner), nn.Linear(dim, inner)
        self.down = nn.Linear(inner, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.cell(self.ln1(x))
        h = self.gn(h.reshape(-1, h.shape[-1])).reshape(h.shape)
        x = x + self.drop(h)
        y = self.ln2(x)
        return x + self.drop(self.down(self.up_a(y) * F.gelu(self.up_b(y))))

In [ ]:
def mlstm_parallel(q, k, v, i_pre, f_pre, n_heads):
    """
    mLSTM en su forma PARALELA estabilizada (Apéndice A.3, Ec. 79-86): procesa todos los
    instantes a la vez con una matriz T x T, como la atención, pero con compuertas.

        q, k, v      : (B, T, D)        D = n_heads * dh
        i_pre, f_pre : (B, T, n_heads)  pre-activaciones de las compuertas de entrada y olvido
        -> h̃         : (B, T, D)

    Es equivalente a la recurrencia  C_t = f_t C_{t-1} + i_t v_t k_tᵀ,  n_t = f_t n_{t-1} + i_t k_t,
    h̃_t = C_t q_t / max(|n_tᵀ q_t|, 1)  (Ec. 19-21), y causal: el instante i solo usa j <= i.
    """
    B, T, D = q.shape
    dh = D // n_heads
    q, k, v = [x.view(B, T, n_heads, dh).transpose(1, 2) for x in (q, k, v)]   # (B, H, T, dh)
    i_pre, f_pre = i_pre.transpose(1, 2), f_pre.transpose(1, 2)                # (B, H, T)

    log_f = F.logsigmoid(f_pre)
    cum = torch.cumsum(log_f, dim=-1)
    log_F = cum.unsqueeze(-1) - cum.unsqueeze(-2)              # [i, j] = sum_{k=j+1..i} log f_k   (Ec. 101)
    causal = torch.ones(T, T, dtype=torch.bool, device=q.device).tril()
    log_D = (log_F + i_pre.unsqueeze(-2)).masked_fill(~causal, float("-inf"))  # Ec. 79: log F + Ĩ
    m = log_D.amax(dim=-1, keepdim=True)                       # Ec. 80: estabilizador por fila
    D_mat = torch.exp(log_D - m)                               # Ec. 81
    C = (q @ k.transpose(-2, -1)) / math.sqrt(dh) * D_mat      # Ec. 82: scores qkᵀ/√d con compuertas
    n = torch.maximum(C.sum(dim=-1, keepdim=True).abs(), torch.exp(-m))   # Ec. 83-84: normalizador
    h = (C / n) @ v                                            # Ec. 85-86
    return h.transpose(1, 2).reshape(B, T, D)


def mlstm_recurrent(q, k, v, i_pre, f_pre, n_heads):
    """La misma mLSTM en forma RECURRENTE (Ec. 19-21, estabilizada). Solo se usa para verificar."""
    B, T, D = q.shape
    dh = D // n_heads
    q, k, v = [x.view(B, T, n_heads, dh) for x in (q, k, v)]
    k = k / math.sqrt(dh)                                      # Ec. 23: k escalado por 1/√d
    C = q.new_zeros(B, n_heads, dh, dh)
    n = q.new_zeros(B, n_heads, dh)
    m = q.new_full((B, n_heads), float("-inf"))
    out = []
    for t in range(T):
        log_f = F.logsigmoid(f_pre[:, t])
        m_new = torch.maximum(log_f + m, i_pre[:, t])
        i_g = torch.exp(i_pre[:, t] - m_new)[..., None]
        f_g = torch.exp(log_f + m - m_new)[..., None]
        C = f_g[..., None] * C + i_g[..., None] * (v[:, t, :, :, None] @ k[:, t, :, None, :])  # C += i v kᵀ
        n = f_g * n + i_g * k[:, t]
        num = (C @ q[:, t, :, :, None]).squeeze(-1)
        den = torch.maximum((n * q[:, t]).sum(-1, keepdim=True).abs(), torch.exp(-m_new)[..., None])
        out.append(num / den)
        m = m_new
    return torch.stack(out, dim=1).reshape(B, T, D)

In [ ]:
class mLSTMBlock(nn.Module):
    """
    Bloque residual mLSTM con "pre up-projection" (Fig. 11 del paper):

        x -> LayerNorm -> proyección x2 en dos ramas:
             rama celda : conv causal (k=4) + Swish -> q, k ; (sin conv) -> v ; compuertas i, f
                          -> mLSTM -> GroupNorm -> + skip aprendible
             rama salida: Swish  (compuerta de salida externa, por componente)
          -> producto de ambas ramas -> proyección de vuelta a dim -> (+x)
    """

    def __init__(self, dim: int, n_heads: int, dropout: float, proj_factor: int = 2,
                 conv_kernel: int = 4, qkv_block: int = 4):
        super().__init__()
        inner = proj_factor * dim
        self.n_heads, self.conv_kernel = n_heads, conv_kernel
        self.ln = nn.LayerNorm(dim)
        self.up_cell, self.up_gate = nn.Linear(dim, inner), nn.Linear(dim, inner)
        self.conv = nn.Conv1d(inner, inner, conv_kernel, groups=inner)       # por dimensión (depthwise)
        self.q = BlockDiagonalLinear(inner, inner // qkv_block)
        self.k = BlockDiagonalLinear(inner, inner // qkv_block)
        self.v = BlockDiagonalLinear(inner, inner // qkv_block)
        self.igate, self.fgate = nn.Linear(3 * inner, n_heads), nn.Linear(3 * inner, n_heads)
        with torch.no_grad():
            self.fgate.bias.copy_(torch.linspace(3.0, 6.0, n_heads))         # forget gate: al inicio recuerda
            nn.init.normal_(self.igate.bias, 0.0, 0.1)                       # input gate: N(0, 0.1) (Tabla 2)
        self.gn = nn.GroupNorm(n_heads, inner)
        self.skip = nn.Parameter(torch.ones(inner))                          # "learnable skip" de la Fig. 11
        self.down = nn.Linear(inner, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.ln(x)
        x_cell, x_gate = self.up_cell(y), self.up_gate(y)
        # Convolución CAUSAL: se rellena solo a la izquierda, así el instante t no ve t+1, t+2, ...
        x_conv = F.silu(self.conv(F.pad(x_cell.transpose(1, 2), (self.conv_kernel - 1, 0))).transpose(1, 2))
        q, k, v = self.q(x_conv), self.k(x_conv), self.v(x_cell)
        qkv = torch.cat([q, k, v], dim=-1)
        h = mlstm_parallel(q, k, v, self.igate(qkv), self.fgate(qkv), self.n_heads)
        h = self.gn(h.reshape(-1, h.shape[-1])).reshape(h.shape) + self.skip * x_conv
        return x + self.drop(self.down(h * F.silu(x_gate)))

In [ ]:
class xLSTMAutoRegressive(nn.Module):
    """
    g_ar alternativo a la GRU, con la MISMA interfaz que AutoRegressive del notebook 01:

        z: (B, T, z_dim) -> c: (B, T, c_dim)

    Proyecta z_t a c_dim y apila `n_blocks` bloques xLSTM residuales (todos sLSTM o todos mLSTM),
    como la GRU apila 2 capas. Todo es causal: c_t solo depende de z_1..z_t.
    """

    def __init__(self, cfg: CPCConfig, kind: str, n_blocks: int = 2, n_heads: int = 4):
        super().__init__()
        assert kind in ("sLSTM", "mLSTM")
        Block = sLSTMBlock if kind == "sLSTM" else mLSTMBlock
        self.proj = nn.Linear(cfg.z_dim, cfg.c_dim)
        self.blocks = nn.ModuleList([Block(cfg.c_dim, n_heads, cfg.ar_dropout) for _ in range(n_blocks)])
        self.ln_out = nn.LayerNorm(cfg.c_dim)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.proj(z)
        for block in self.blocks:
            x = block(x)
        return self.ln_out(x)

In [ ]:
def build_model(kind: str, horizons: tuple, seed: int, device=None) -> CPCModel:
    """CPC completo con el g_ar elegido. La semilla fija los pesos iniciales (misma semilla = misma red)."""
    cfg = CPCConfig(n_channels=N_CH, n_pred_steps=len(horizons), n_anchors=4, pred_horizons=tuple(horizons))
    torch.manual_seed(seed)
    model = CPCModel(cfg)
    if kind != "GRU":
        model.ar = xLSTMAutoRegressive(cfg, kind)
    return model.to(device or DEVICE)


def summarize(c: torch.Tensor, representation: str) -> torch.Tensor:
    """c: (B, T, c_dim) -> vector por ventana."""
    if representation == "c_T":
        return c[:, -1]
    if representation == "cola":
        return c[:, -TAIL:].mean(dim=1)
    if representation == "media":
        return c.mean(dim=1)
    if representation == "c_T+media":
        return torch.cat([c[:, -1], c.mean(dim=1)], dim=1)
    raise ValueError(representation)

In [ ]:
# Clasificador para el ajuste fino (copiado de la sección 10 del 06)
class CPCClassifier(nn.Module):
    def __init__(self, cpc_model, representation, n_classes=len(LABELS)):
        super().__init__()
        self.encoder, self.ar, self.representation = cpc_model.encoder, cpc_model.ar, representation
        dim = cpc_model.cfg.c_dim * (2 if representation == "c_T+media" else 1)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.2), nn.Linear(dim, n_classes))

    def forward(self, x):
        z = self.encoder(x).transpose(1, 2)
        return self.head(summarize(self.ar(z), self.representation))

# 4. Herramientas por pliegue

- **`prepare_fold(sujeto)`:** reajusta el escalador con los sujetos de entrenamiento del pliegue, escala toda la serie y arma los conjuntos:
  - **CPC:** recortes aleatorios de los ensayos de entrenamiento;
  - **sonda:** muestra estratificada de entrenamiento;
  - **ajuste fino:** todas las ventanas de entrenamiento;
  - **evaluación:** todas las ventanas del sujeto que quedó fuera.
- **`train_cpc_fixed`:** preentrena CPC un número fijo de épocas. Es el mismo bucle del 06, sin parada temprana.
- **`finetune_fixed`:** ajusta el modelo completo con entropía cruzada ponderada por clase, un número fijo de épocas.
- **`fit_probe`:** la misma sonda del 06 (`StandardScaler` + regresión logística balanceada), que se usa también con las características tabulares.

In [ ]:
def fit_fold_scaler(train_subjects):
    """clip por cuantiles + log1p + RobustScaler, por canal, ajustado sólo con los sujetos de entrenamiento."""
    rows = trials[trials["subject"].isin(train_subjects)]
    idx = np.concatenate([np.arange(r.offset, r.offset + r.n_steps) for r in rows.itertuples()])
    rng = np.random.default_rng(SEED)
    sample = signals_raw[rng.choice(idx, size=min(SCALER_FIT_MAX, len(idx)), replace=False)]
    lower, upper = np.quantile(sample, SCALER_Q, axis=0)
    robust = RobustScaler(quantile_range=(25.0, 75.0)).fit(np.log1p(np.clip(sample, lower, upper)))
    return {"lower": lower, "upper": upper, "center": robust.center_, "scale": robust.scale_}


def stratified_cap(frame, cap):
    if cap is None or len(frame) <= cap:
        return frame
    return frame.groupby("target", group_keys=False).sample(frac=cap / len(frame), random_state=SEED)


def starts_of(frame):
    return offsets[frame["trial_idx"].to_numpy()] + frame["start_step"].to_numpy()


def make_set(frame):
    frame = frame.reset_index(drop=True)
    return {"frame": frame, "starts": starts_of(frame), "y": frame["target"].to_numpy()}


def get_windows(starts):
    """Posiciones absolutas de inicio -> tensor (B, canales, WIN) en el dispositivo."""
    idx = torch.as_tensor(np.asarray(starts), device=DEVICE)[:, None] + WIN_RANGE
    return SIGNALS_T[:, idx].permute(1, 0, 2)


class CPCBatchSampler:
    """Inicios de ventana para CPC con los sujetos de entrenamiento del pliegue (mismo criterio que el 06)."""

    def __init__(self, subjects, batch_mode, seed, batch_size=BATCH_SIZE):
        rows = trials[trials["subject"].isin(subjects) & (trials["n_steps"] >= WIN)]
        self.starts = np.concatenate([np.arange(r.offset, r.offset + r.n_steps - WIN + 1) for r in rows.itertuples()])
        subj = np.concatenate([np.full(r.n_steps - WIN + 1, r.subject) for r in rows.itertuples()])
        self.by_subject = {s: self.starts[subj == s] for s in np.unique(subj)}
        sizes = np.array([len(v) for v in self.by_subject.values()], dtype=float)
        self.subject_p = sizes / sizes.sum()
        self.batch_mode, self.batch_size, self.rng = batch_mode, batch_size, np.random.default_rng(seed)

    def batch(self):
        if self.batch_mode == "mezclado":
            return self.rng.choice(self.starts, self.batch_size, replace=False)
        subject = self.rng.choice(list(self.by_subject), p=self.subject_p)
        pool = self.by_subject[subject]
        return self.rng.choice(pool, self.batch_size, replace=len(pool) < self.batch_size)


def prepare_fold(held_out):
    """Escala la serie con los sujetos de entrenamiento del pliegue y arma sus conjuntos."""
    global SIGNALS_T, FOLD
    train_subjects = [s for s in DEV_SUBJECTS if s != held_out]
    scaler = fit_fold_scaler(train_subjects)
    scaled = (np.log1p(np.clip(signals_raw, scaler["lower"], scaler["upper"])) - scaler["center"]) / scaler["scale"]
    if CHANNEL_MODE == "pares4":
        scaled = scaled.reshape(-1, 4, 2).mean(axis=2)
    SIGNALS_T = torch.from_numpy(np.ascontiguousarray(scaled.T.astype(np.float32))).to(DEVICE)
    sup_train = sup_index[sup_index["subject"].isin(train_subjects)]
    FOLD = {
        "held_out": held_out, "train_subjects": train_subjects, "scaler": scaler,
        "probe_train": make_set(stratified_cap(sup_train, PROBE_TRAIN_MAX)),
        "ft_train": make_set(stratified_cap(sup_train, FT_TRAIN_CAP)),
        "eval": make_set(stratified_cap(sup_index[sup_index["subject"] == held_out], EVAL_CAP)),
        "n_steps": min(len(cpc_index[cpc_index["subject"].isin(train_subjects)]) // BATCH_SIZE, STEPS_CAP or 10**9),
    }
    counts = np.bincount(FOLD["ft_train"]["y"], minlength=len(LABELS))
    FOLD["class_weights"] = torch.tensor(counts.sum() / (len(LABELS) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    return FOLD


def metrics(y_true, y_pred):
    ids = list(range(len(LABELS)))
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_per_class": f1_score(y_true, y_pred, labels=ids, average=None, zero_division=0).tolist(),
        "confusion": confusion_matrix(y_true, y_pred, labels=ids, normalize="true").tolist(),
    }


def fit_probe(X, y):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(X, y)


@torch.no_grad()
def extract_rep(model, starts, rep, batch=512):
    model.eval()
    out = []
    for s in range(0, len(starts), batch):
        _, c = model.encode(get_windows(starts[s:s + batch]))
        out.append(summarize(c, rep).float().cpu().numpy())
    return np.concatenate(out)


def probe_metrics(model, rep):
    clf = fit_probe(extract_rep(model, FOLD["probe_train"]["starts"], rep), FOLD["probe_train"]["y"])
    return metrics(FOLD["eval"]["y"], clf.predict(extract_rep(model, FOLD["eval"]["starts"], rep)))


def train_cpc_fixed(cfg, seed=SEED):
    model = build_model(cfg["kind"], cfg["horizons"], seed)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sampler = CPCBatchSampler(FOLD["train_subjects"], cfg["batch_mode"], seed)
    torch.manual_seed(seed)
    history = []
    for epoch in range(1, cfg["cpc_epochs"] + 1):
        model.train()
        t0, run_loss = time.time(), 0.0
        for _ in range(FOLD["n_steps"]):
            loss, _, _ = model.info_nce(get_windows(sampler.batch()))
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            run_loss += loss.item()
        history.append({"epoch": epoch, "train_loss": run_loss / FOLD["n_steps"], "secs": time.time() - t0})
        print(f"      CPC época {epoch:2d}/{cfg['cpc_epochs']} | InfoNCE {history[-1]['train_loss']:.3f} | {history[-1]['secs']:.0f} s")
    return model.eval(), history


@torch.no_grad()
def predict(clf_model, starts, batch=512):
    clf_model.eval()
    return np.concatenate([clf_model(get_windows(starts[s:s + batch])).argmax(1).cpu().numpy()
                           for s in range(0, len(starts), batch)])


def finetune_fixed(base, cfg, variant, seed=SEED):
    torch.manual_seed(seed)
    model = CPCClassifier(base, cfg["rep"]).to(DEVICE)
    opt = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()) + list(model.ar.parameters()), "lr": FINETUNE_LR_BACKBONE},
        {"params": model.head.parameters(), "lr": FINETUNE_LR_HEAD},
    ], weight_decay=1e-4)
    rng = np.random.default_rng(seed)
    data = FOLD["ft_train"]
    y_all = torch.as_tensor(data["y"], device=DEVICE)
    for epoch in range(1, cfg["ft_epochs"][variant] + 1):
        model.train()
        order, run_loss = rng.permutation(len(data["y"])), 0.0
        for s in range(0, len(order), FINETUNE_BATCH):
            idx = order[s:s + FINETUNE_BATCH]
            loss = F.cross_entropy(model(get_windows(data["starts"][idx])), y_all[idx], weight=FOLD["class_weights"])
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            run_loss += loss.item() * len(idx)
        print(f"      ajuste fino {variant} época {epoch}/{cfg['ft_epochs'][variant]} | loss {run_loss / len(order):.3f}")
    return metrics(FOLD["eval"]["y"], predict(model, FOLD["eval"]["starts"]))


def features_for(es):
    return feat_lookup.loc[list(zip(es["frame"]["trial_idx"], es["frame"]["end_step"]))].to_numpy(dtype=np.float32)

# 5. Verificación rápida

Antes de gastar horas de cómputo se comprueba, en el primer pliegue y para cada configuración:

1. **Causalidad:** perturbar la entrada desde `T/2` no cambia nada de `c` antes de `T/2`.
2. **Dispositivo:** en modo evaluación, el modelo da lo mismo en el dispositivo (MPS) que en CPU.
3. **Sin fuga entre pliegues:** el sujeto evaluado no aparece en los conjuntos de entrenamiento, y el escalador no usó su señal.

In [ ]:
fold = prepare_fold(HELD_OUT[0])
assert fold["held_out"] not in set(fold["probe_train"]["frame"]["subject"]) | set(fold["ft_train"]["frame"]["subject"])
assert set(fold["eval"]["frame"]["subject"]) == {fold["held_out"]}
rows = []
for name, cfg in CONFIGS.items():
    model = build_model(cfg["kind"], cfg["horizons"], SEED).eval()
    x = torch.randn(4, N_CH, WIN, device=DEVICE)
    x_mod = x.clone()
    x_mod[:, :, WIN // 2:] += torch.randn_like(x_mod[:, :, WIN // 2:])
    with torch.no_grad():
        _, c = model.encode(x)
        _, c_mod = model.encode(x_mod)
        cpu_model = build_model(cfg["kind"], cfg["horizons"], SEED, device="cpu").eval()
        device_gap = (c.cpu() - cpu_model.encode(x.cpu())[1]).abs().max().item()
    past = (c[:, :WIN // 2] - c_mod[:, :WIN // 2]).abs().max().item()
    assert past == 0.0, f"{name}: fuga temporal"
    rows.append({"configuración": name, "fuga antes de T/2": past, "dif. dispositivo vs CPU": device_gap})
    del model, cpu_model
display(pd.DataFrame(rows).set_index("configuración"))
print(f"Pliegue de prueba: sujeto {fold['held_out']} | entrenamiento {fold['train_subjects']} | "
      f"{len(fold['eval']['y']):,} ventanas de evaluación | {fold['n_steps']} pasos CPC por época")

# 6. Ejecución de los pliegues

Para cada sujeto que queda fuera se calculan:

- **`tabular`:** las 202 características del avance 2 con la sonda lineal. Es la referencia sin aprendizaje profundo.
- Por cada configuración:
  - **`sin_entrenar`:** la misma red con pesos aleatorios, más la sonda lineal;
  - **`sonda`:** CPC preentrenado en este pliegue, congelado, más la sonda lineal;
  - **`preentrenado`:** ajuste fino partiendo de ese CPC;
  - **`desde_cero`:** ajuste fino partiendo de pesos aleatorios. Es el control que dice si CPC aporta cuando también se entrena con etiquetas.

Cada resultado se guarda como `.json` en `RUN_DIR`; si ya existe, se salta.

In [ ]:
def result_path(held_out, view):
    return RUN_DIR / f"s{held_out:02d}_{view}.json"


def save_result(held_out, view, m, minutes, extra=None):
    payload = {"held_out": held_out, "view": view, "minutes": minutes, "metrics": m, **(extra or {})}
    result_path(held_out, view).write_text(json.dumps(payload))
    print(f"   {view:<28} F1 {m['f1_macro']:.3f} | bal. acc {m['balanced_accuracy']:.3f} | {minutes:.1f} min")


def run_fold(held_out):
    todo = [("tabular", None)] + [(f"{n}_{v}", n) for n in CONFIGS for v in VARIANTS]
    if not RERUN and all(result_path(held_out, view).exists() for view, _ in todo):
        print(f"Sujeto {held_out:2d}: completo (resultados guardados)")
        return
    print(f"Sujeto {held_out:2d}: preparando pliegue...")
    prepare_fold(held_out)
    if RERUN or not result_path(held_out, "tabular").exists():
        t0 = time.time()
        clf = fit_probe(features_for(FOLD["probe_train"]), FOLD["probe_train"]["y"])
        save_result(held_out, "tabular", metrics(FOLD["eval"]["y"], clf.predict(features_for(FOLD["eval"]))), (time.time() - t0) / 60)
    for name, cfg in CONFIGS.items():
        views = {v: f"{name}_{v}" for v in VARIANTS}
        if not RERUN and all(result_path(held_out, view).exists() for view in views.values()):
            continue
        if RERUN or not result_path(held_out, views["sin_entrenar"]).exists():
            t0 = time.time()
            save_result(held_out, views["sin_entrenar"], probe_metrics(build_model(cfg["kind"], cfg["horizons"], SEED), cfg["rep"]),
                        (time.time() - t0) / 60)
        cpc_ckpt = RUN_DIR / f"s{held_out:02d}_{name}_cpc.pt"
        if cpc_ckpt.exists() and not RERUN:
            cpc_model = build_model(cfg["kind"], cfg["horizons"], SEED)
            cpc_model.load_state_dict(torch.load(cpc_ckpt, map_location=DEVICE)["state_dict"])
            cpc_model.eval()
        else:
            t0 = time.time()
            print(f"   {name}: preentrenando CPC ({cfg['cpc_epochs']} épocas)")
            cpc_model, history = train_cpc_fixed(cfg)
            torch.save({"state_dict": cpc_model.state_dict(), "history": history, "scaler": {k: v.tolist() for k, v in FOLD["scaler"].items()},
                        "config": {k: v for k, v in cfg.items()}}, cpc_ckpt)
            print(f"      CPC listo en {(time.time() - t0) / 60:.1f} min")
        if RERUN or not result_path(held_out, views["sonda"]).exists():
            t0 = time.time()
            save_result(held_out, views["sonda"], probe_metrics(cpc_model, cfg["rep"]), (time.time() - t0) / 60)
        for variant in ("preentrenado", "desde_cero"):
            if RERUN or not result_path(held_out, views[variant]).exists():
                t0 = time.time()
                base = copy.deepcopy(cpc_model) if variant == "preentrenado" else build_model(cfg["kind"], cfg["horizons"], SEED)
                save_result(held_out, views[variant], finetune_fixed(base, cfg, variant), (time.time() - t0) / 60)
                del base
        del cpc_model
        if DEVICE.type == "mps":
            torch.mps.empty_cache()


t_total = time.time()
for i, held_out in enumerate(HELD_OUT, 1):
    t0 = time.time()
    run_fold(held_out)
    elapsed = (time.time() - t_total) / 60
    print(f"   [{i}/{len(HELD_OUT)}] pliegue en {(time.time() - t0) / 60:.1f} min | total {elapsed:.0f} min")

# 7. Resultados por sujeto

La tabla resume, para cada vista, el F1 macro de los sujetos que quedaron fuera:

- **media:** la estimación del desempeño con una persona nueva;
- **desviación estándar:** cuánto varía de una persona a otra;
- **mínimo y máximo:** el peor y el mejor sujeto.

La figura muestra cada sujeto como un punto, y una línea por sujeto une sus valores en las distintas vistas. Si las líneas son casi paralelas, la diferencia entre vistas es consistente aunque los sujetos sean muy distintos entre sí.

In [ ]:
rows = []
for path in sorted(RUN_DIR.glob("s*_*.json")):
    r = json.loads(path.read_text())
    m = r["metrics"]
    rows.append({"sujeto": r["held_out"], "vista": r["view"], "F1": m["f1_macro"], "bal_acc": m["balanced_accuracy"],
                 "minutos": r["minutes"], **{f"F1 {l}": f for l, f in zip(LABELS, m["f1_per_class"])}})
loso = pd.DataFrame(rows)
loso.to_csv(SUMMARY_CSV, index=False)
print(f"{loso['sujeto'].nunique()} pliegues con resultados | guardado en {SUMMARY_CSV.relative_to(REPO_DIR)}")

summary = (loso.groupby("vista")["F1"].agg(["mean", "std", "min", "max", "count"])
           .join(loso.groupby("vista")["bal_acc"].mean().rename("bal_acc media"))
           .sort_values("mean", ascending=False))
display(summary.round(3))

wide = loso.pivot(index="sujeto", columns="vista", values="F1")[summary.index]
fig, ax = plt.subplots(figsize=(12, 4.5))
for subject, row in wide.iterrows():
    ax.plot(range(len(row)), row.to_numpy(), color="#c7c7c7", lw=0.8, zorder=1)
    ax.text(len(row) - 0.9, row.iloc[-1], str(subject), fontsize=7, color="#7f7f7f", va="center")
for j, view in enumerate(wide.columns):
    ax.scatter([j] * len(wide), wide[view], s=18, color="#2a78d6", zorder=2)
    ax.hlines(wide[view].mean(), j - 0.3, j + 0.3, color="black", lw=2, zorder=3)
ax.set(xticks=range(len(wide.columns)), ylabel="F1 macro (sujeto que quedó fuera)",
       title="LOSO: cada punto es un sujeto; la barra negra, la media")
ax.set_xticklabels(wide.columns, rotation=30, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

display(loso.groupby("vista")[[f"F1 {l}" for l in LABELS]].mean().loc[summary.index].round(3))

## 7.1 Comparaciones pareadas

Cada sujeto se compara **consigo mismo** en dos vistas, lo que elimina la variación entre personas. Para cada pareja se reporta:

- **diferencia media de F1** entre las dos vistas;
- **cuántos sujetos mejoran:** si la ventaja aparece en casi todos, es consistente;
- **prueba de rangos con signo de Wilcoxon:** no supone normalidad y es adecuada para pocas observaciones pareadas. Con 12 sujetos, un p < 0.05 requiere que la mejora se repita en la gran mayoría de ellos.

Preguntas que responde:

1. **¿Aporta CPC sin etiquetas?** Sonda sobre CPC contra la misma red sin entrenar.
2. **¿Aporta CPC cuando también hay ajuste fino?** Preentrenado contra desde cero.
3. **¿Supera a las características manuales?** El mejor ajuste fino contra la línea base tabular.
4. **¿Qué arquitectura conviene?** mLSTM contra sLSTM, ambos con ajuste fino preentrenado.

In [ ]:
def paired(a, b):
    both = wide[[a, b]].dropna()
    diff = both[a] - both[b]
    p = wilcoxon(both[a], both[b]).pvalue if len(both) >= 6 and (diff != 0).any() else np.nan
    return {"A": a, "B": b, "sujetos": len(both), "dif. media (A − B)": diff.mean(),
            "sujetos donde A > B": int((diff > 0).sum()), "p (Wilcoxon)": p}


pairs = []
for name in CONFIGS:
    pairs += [paired(f"{name}_sonda", f"{name}_sin_entrenar"), paired(f"{name}_preentrenado", f"{name}_desde_cero")]
best_ft = max((f"{n}_preentrenado" for n in CONFIGS), key=lambda v: wide[v].mean() if v in wide else -1)
pairs.append(paired(best_ft, "tabular"))
names = list(CONFIGS)
if len(names) >= 2:
    pairs.append(paired(f"{names[0]}_preentrenado", f"{names[1]}_preentrenado"))
display(pd.DataFrame(pairs).set_index(["A", "B"]).round(3))

# 8. Cómo leer los resultados

- **El número principal** es la media ± desviación del F1 de la vista elegida. Reemplaza al F1 de validación del 06 (sujetos 11 y 12) como estimación de desempeño con una persona nueva.
- **La desviación entre sujetos** es la incertidumbre real del modelo. Si es grande (por ejemplo, ±0.06), un sujeto nuevo puede quedar muy por encima o muy por debajo de la media. Conviene reportarla siempre junto con la media.
- **Diferencias entre vistas:** sólo se consideran establecidas si mejoran en la mayoría de los sujetos (columna "sujetos donde A > B") y el p de Wilcoxon es pequeño. Una diferencia de 0.02 que aparece en 6 de 12 sujetos no es concluyente.
- **Sujetos 11 y 12:** las épocas fijas se eligieron con ellos en el 06, así que sus pliegues pueden salir algo optimistas. Si sus F1 destacan claramente sobre el resto, conviene reportar también la media de los otros 10.
- **Decisión final:** se elige la vista con mejor media y ventaja consistente. Sólo entonces se abre la prueba (sujetos 13 y 14), entrenando con los 12 sujetos de desarrollo, una única vez.